In [ ]:
import sys, os
import glob
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from cmdstanpy import from_csv
from utils.generate_data import generate_Friedman_data
from utils.sparsity import forward_pass_tanh
from utils.sparsity_deep import forward_pass_tanh_deep, build_posterior_mask

## Cell 1 — Load fits

All models from `results/regression/single_layer_H16/tanh/friedman/` plus DHS L3 from `results_remote/`.

In [ ]:
_KEY = re.compile(r"Friedman_N(\d+)_p\d+_sigma([\d.]+)_seed(\d+)")

RESULTS_DIR  = "results/regression/single_layer_H16/tanh/friedman"
REMOTE_DIR   = "results_remote"
DATA_DIR     = "datasets/friedman"

# (display label) -> (subdir inside RESULTS_DIR, num_hidden_layers, is_remote)
ARCH_MAP = {
    "Gauss L1":  ("gaussian_tanh_H16_L1",                          1, False),
    "Gauss L2":  ("gaussian_tanh_H16_L2",                          2, False),
    "Gauss L3":  ("gaussian_tanh_H16_L3",                          3, False),
    "RHS L1":    ("regularized_horseshoe_tanh_H16_L1",             1, False),
    "RHS L2":    ("regularized_horseshoe_tanh_H16_L2",             2, False),
    "RHS L3":    ("regularized_horseshoe_tanh_H16_L3",             3, False),
    "DHS L1":    ("dirichlet_horseshoe_tanh_nodewise_H16_L1",      1, False),
    "DHS L2":    ("dirichlet_horseshoe_tanh_nodewise_H16_L2",      2, False),
    "DHS L3":    ("dirichlet_horseshoe_tanh_nodewise_H16_L3",      3, True),
    "DST L1":    ("dirichlet_student_t_tanh_nodewise_H16_L1",      1, False),
    "DST L2":    ("dirichlet_student_t_tanh_nodewise_H16_L2",      2, False),
    "DST L3":    ("dirichlet_student_t_tanh_nodewise_H16_L3",      3, False),
}

configs = sorted(
    f.replace(".npz", "")
    for f in os.listdir(DATA_DIR) if f.endswith(".npz")
)

def load_fit(subdir, config, is_remote):
    root = REMOTE_DIR if is_remote else RESULTS_DIR
    pattern = os.path.join(root, subdir, config, "chain_*.csv")
    files = sorted(glob.glob(pattern))
    if not files:
        return None
    return from_csv(files, method="sample")

# fits[arch_label][config] = CmdStanPy fit object (or None if missing)
fits = {}
for label, (subdir, n_layers, is_remote) in ARCH_MAP.items():
    fits[label] = {}
    for config in configs:
        fit = load_fit(subdir, config, is_remote)
        if fit is not None:
            fits[label][config] = fit
    print(f"{label:12s}: {len(fits[label])} configs loaded")

## Cell 2 — Posterior-prune sparsity sweep

Mask is built on `W_1` (input weights) using E|w| across posterior draws.
RMSE is computed on a fresh large test set (N_test=2000) in original y scale.

In [ ]:
SPARSITY_LEVELS = [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
N_TEST = 2000

def get_weights(fit, n_hidden_layers):
    """Return (W1, W_int_all, W_L, b_hidden, b_out) for one fit."""
    W1  = fit.stan_variable("W_1")         # (S, P, H)
    W_L = fit.stan_variable("W_L")         # (S, H, O)
    b_h = fit.stan_variable("hidden_bias") # (S, n_layers, H)
    b_o = fit.stan_variable("output_bias") # (S, O) or (S, 1)
    W_int_all = None
    if n_hidden_layers > 1:
        W_int_all = fit.stan_variable("W_internal")  # (S, n_internal, H_in, H_out)
    return W1, W_int_all, W_L, b_h, b_o

def predict_masked(fit, n_hidden_layers, mask_W1, X_test):
    """Return posterior-mean prediction (N_test,) with W1 masked."""
    W1, W_int_all, W_L, b_h, b_o = get_weights(fit, n_hidden_layers)
    S = W1.shape[0]
    y_hats = np.zeros((S, X_test.shape[0]))
    for s in range(S):
        w1s = W1[s] * mask_W1
        if n_hidden_layers == 1:
            y_hats[s] = forward_pass_tanh(
                X_test, w1s, b_h[s, 0], W_L[s], b_o[s].reshape(-1)
            ).squeeze()
        else:
            # W_int_all is (S, n_internal, H_in, H_out) — axis 1 is layer index
            W_internals = [W_int_all[s, k, :, :] for k in range(W_int_all.shape[1])]
            y_hats[s] = forward_pass_tanh_deep(
                X_test, w1s, W_internals, W_L[s], b_h[s], b_o[s].reshape(-1)
            ).squeeze()
    return y_hats.mean(axis=0)

rows = []
for label, (subdir, n_layers, _) in ARCH_MAP.items():
    for config, fit in fits[label].items():
        m = _KEY.match(config)
        N, sigma, seed = int(m.group(1)), float(m.group(2)), int(m.group(3))

        _, _, y_tr, _        = generate_Friedman_data(N=N, D=10, sigma=sigma, seed=seed)
        y_mean, y_std        = y_tr.mean(), y_tr.std()
        _, X_te, _, y_te_raw = generate_Friedman_data(N=N_TEST, D=10, sigma=sigma, seed=seed + 999)
        y_te = (y_te_raw - y_mean) / y_std

        W1_samples = fit.stan_variable("W_1")
        for q in SPARSITY_LEVELS:
            mask  = build_posterior_mask(W1_samples, q)
            y_pred = predict_masked(fit, n_layers, mask, X_te)
            rmse  = float(np.sqrt(np.mean((y_pred - y_te) ** 2))) * y_std
            rows.append(dict(label=label, config=config, N=N, sigma=sigma,
                             seed=seed, sparsity=q, rmse=rmse))

df_sparsity = pd.DataFrame(rows)

df_agg = (
    df_sparsity.groupby(["label", "N", "sparsity"])["rmse"]
    .agg(center="mean", spread="std")
    .reset_index()
)
print(df_agg.head())

## Cell 3 — Posterior-prune RMSE curves (L1 models, grouped by N)

Addresses R1: show posterior-prune sparsity trade-off systematically across N.

In [ ]:
L1_LABELS  = ["Gauss L1", "RHS L1", "DHS L1", "DST L1"]
COLORS_L1  = {"Gauss L1": "C0", "RHS L1": "C1", "DHS L1": "C2", "DST L1": "C3"}
ABBR       = {"Gauss L1": "Gauss", "RHS L1": "RHS", "DHS L1": "DHS", "DST L1": "DST"}
Ns         = [100, 200, 500]

fig, axes = plt.subplots(1, 3, figsize=(13, 4), sharey=False)
for ax, N in zip(axes, Ns):
    sub = df_agg[(df_agg["N"] == N) & (df_agg["label"].isin(L1_LABELS))]
    for label in L1_LABELS:
        g = sub[sub["label"] == label].sort_values("sparsity")
        ax.plot(g["sparsity"], g["center"], lw=2.5, marker="o",
                color=COLORS_L1[label], label=ABBR[label])
    ax.set_title(f"N={N}", fontsize=16)
    ax.set_xlabel("Sparsity", fontsize=13)
    ax.set_xticks(SPARSITY_LEVELS[::2])
    ax.grid(True, linestyle="--", linewidth=0.5)
    ax.tick_params(labelsize=12)
axes[0].set_ylabel("RMSE (original scale)", fontsize=13)
handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="upper right", frameon=False, fontsize=12)
plt.tight_layout()
plt.show()

## Cell 4 — Dirichlet allocation (φ_data) entropy: within-node sparsity

`phi_data` is the Dirichlet allocation weight vector for each hidden node — shape `(S, H, P)`.
Each node distributes its variance budget across P incoming weights.
Low entropy → budget concentrated on few inputs → sparse within-node connectivity.
High entropy → budget spread uniformly → no within-node sparsity.

We compare DHS L1 vs DHS L3 to see whether deeper networks also learn concentrated allocation.

In [ ]:
TARGET_N = 200
PHI_ARCHS = {
    "DHS L1": "DHS L1",
    "DHS L3": "DHS L3",
}

target_configs = [c for c in configs if f"_N{TARGET_N}_" in c]

def phi_entropy(phi_samples):
    """
    phi_samples: (S, H, P)  — Dirichlet allocation weights per node.
    Returns (S, H) posterior entropy per node.
    Clipped to avoid log(0).
    """
    phi_c = np.clip(phi_samples, 1e-12, None)
    return -(phi_c * np.log(phi_c)).sum(axis=-1)   # (S, H)

# Collect per-seed posterior-mean entropy across nodes: dict -> (n_seeds, H)
entropy_data = {}
for arch_label in PHI_ARCHS.values():
    seed_entropies = []
    for config in target_configs:
        fit = fits[arch_label].get(config)
        if fit is None:
            continue
        phi = fit.stan_variable("phi_data")          # (S, H, P)
        ent = phi_entropy(phi)                        # (S, H)
        seed_entropies.append(ent.mean(axis=0))       # mean over draws -> (H,)
    if seed_entropies:
        entropy_data[arch_label] = np.stack(seed_entropies)  # (n_seeds, H)

fig, axes = plt.subplots(1, len(entropy_data), figsize=(5 * len(entropy_data), 4), sharey=True)
if len(entropy_data) == 1:
    axes = [axes]

for ax, (arch_label, arr) in zip(axes, entropy_data.items()):
    # Sort nodes by median entropy (ascending = most concentrated first)
    order = np.argsort(np.median(arr, axis=0))
    ax.boxplot(arr[:, order], patch_artist=True,
               boxprops=dict(facecolor="steelblue", alpha=0.6),
               medianprops=dict(color="navy", lw=2),
               showfliers=False)
    # Reference line: uniform entropy = log(P)
    P = fits[arch_label][target_configs[0]].stan_variable("phi_data").shape[-1]
    ax.axhline(np.log(P), color="red", lw=1.5, linestyle="--", label=f"Uniform (log P={np.log(P):.2f})")
    ax.set_title(f"{arch_label}  (N={TARGET_N})", fontsize=14)
    ax.set_xlabel("Node (sorted by median entropy)", fontsize=12)
    ax.set_ylabel("Posterior entropy of φ_data", fontsize=12)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.tick_params(axis="x", labelbottom=False)
    ax.legend(fontsize=10, frameon=False)

plt.suptitle(f"Within-node sparsity: φ_data entropy  (N={TARGET_N})", fontsize=14)
plt.tight_layout()
plt.show()

## Cell 5 — Scaling: baseline RMSE vs depth (sparsity=0)

Addresses R2: shows how each model family scales with depth.
Groups: Gauss / RHS / DHS / DST, each at L1/L2/L3.

In [ ]:
FAMILIES = {
    "Gauss": ["Gauss L1", "Gauss L2", "Gauss L3"],
    "RHS":   ["RHS L1",   "RHS L2",   "RHS L3"],
    "DHS":   ["DHS L1",   "DHS L2",   "DHS L3"],
    "DST":   ["DST L1",   "DST L2",   "DST L3"],
}
FAMILY_COLORS = {"Gauss": "C0", "RHS": "C1", "DHS": "C2", "DST": "C3"}
DEPTH_MARKERS = {"L1": "o", "L2": "s", "L3": "^"}  

df_base = df_agg[df_agg["sparsity"] == 0.0].copy()
df_base["depth"] = df_base["label"].str.extract(r"(L\d)$")
df_base["family"] = df_base["label"].str.extract(r"^(\S+)\s")

plot_Ns = [100, 200, 500]
fig, axes = plt.subplots(1, len(plot_Ns), figsize=(13, 4), sharey=False)

for ax, N in zip(axes, plot_Ns):
    sub = df_base[df_base["N"] == N]
    for family, labels in FAMILIES.items():
        for lbl in labels:
            row = sub[sub["label"] == lbl]
            if row.empty:
                continue
            depth = lbl.split()[-1]
            x_pos = ["L1", "L2", "L3"].index(depth) + 1
            ax.plot(x_pos, row["center"].values[0],
                    marker=DEPTH_MARKERS[depth], color=FAMILY_COLORS[family],
                    markersize=9, lw=0,
                    label=family if depth == "L1" else "_nolegend_")
        # connect depths with a line
        pts = []
        for depth in ["L1", "L2", "L3"]:
            lbl = f"{family} {depth}"
            row = sub[sub["label"] == lbl]
            if not row.empty:
                pts.append((["L1", "L2", "L3"].index(depth) + 1,
                             row["center"].values[0]))
        if pts:
            xs, ys = zip(*pts)
            ax.plot(xs, ys, color=FAMILY_COLORS[family], lw=1.8, alpha=0.5)

    ax.set_title(f"N={N}", fontsize=16)
    ax.set_xticks([1, 2, 3])
    ax.set_xticklabels(["L1", "L2", "L3"], fontsize=13)
    ax.set_xlabel("Depth", fontsize=13)
    ax.grid(axis="y", linestyle="--", alpha=0.4)
    ax.tick_params(labelsize=12)

axes[0].set_ylabel("RMSE (original scale)", fontsize=13)
handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, loc="upper right", frameon=False, fontsize=12)
plt.suptitle("Baseline RMSE vs depth (sparsity=0)", fontsize=14)
plt.tight_layout()
plt.show()

## Cell 6 — Sparsity-robustness vs depth (DHS, N=200)

Do deeper DHS networks degrade more gracefully under pruning?
Ties the scaling and sparsity narratives together.

In [ ]:
FOCUS_N    = 200
DHS_DEPTHS = ["DHS L1", "DHS L2", "DHS L3"]
DEPTH_COLORS = {"DHS L1": "C2", "DHS L2": "C4", "DHS L3": "C5"}
DEPTH_LS     = {"DHS L1": "-", "DHS L2": "--", "DHS L3": ":"}

sub = df_agg[(df_agg["N"] == FOCUS_N) & (df_agg["label"].isin(DHS_DEPTHS))]

fig, ax = plt.subplots(figsize=(6, 4))
for lbl in DHS_DEPTHS:
    g = sub[sub["label"] == lbl].sort_values("sparsity")
    ax.plot(g["sparsity"], g["center"], lw=2.5, marker="o",
            color=DEPTH_COLORS[lbl], linestyle=DEPTH_LS[lbl], label=lbl)
ax.set_xlabel("Sparsity", fontsize=13)
ax.set_ylabel("RMSE (original scale)", fontsize=13)
ax.set_title(f"DHS: sparsity-robustness vs depth  (N={FOCUS_N})", fontsize=13)
ax.set_xticks(SPARSITY_LEVELS[::2])
ax.grid(linestyle="--", alpha=0.4)
ax.legend(frameon=False, fontsize=12)
plt.tight_layout()
plt.show()

## Cell 7 — φ entropy across layers (DHS L3): does within-node sparsity persist through depth?

Heatmap: rows = layer (input, internal-1, internal-2), cols = node index,
color = mean posterior entropy of φ across seeds.
Low entropy (dark) → node concentrates its budget on few incoming weights → sparse.
Sorted by input-layer entropy so the pattern across layers is easy to read.

In [ ]:
HEAT_N = 200
heat_configs = [c for c in configs if f"_N{HEAT_N}_" in c]

# phi_data:     (S, H, P_in)        — input layer Dirichlet weights
# phi_internal: (S, n_internal, H, H) — internal layers Dirichlet weights
# Entropy per node: -(phi * log phi).sum(axis=-1)  -> (S, H) or (S, n_int, H)

ent_input_seeds    = []   # (n_seeds, H)
ent_internal_seeds = None
n_internal = None

for config in heat_configs:
    fit = fits["DHS L3"].get(config)
    if fit is None:
        continue
    phi_in  = fit.stan_variable("phi_data")      # (S, H, P)
    phi_int = fit.stan_variable("phi_internal")  # (S, n_internal, H, H)

    ent_in  = phi_entropy(phi_in)                # (S, H)
    ent_input_seeds.append(ent_in.mean(axis=0))  # (H,)

    if n_internal is None:
        n_internal = phi_int.shape[1]
        ent_internal_seeds = [[] for _ in range(n_internal)]
    for k in range(n_internal):
        ent_k = phi_entropy(phi_int[:, k, :, :]) # (S, H)
        ent_internal_seeds[k].append(ent_k.mean(axis=0))  # (H,)

# Average over seeds
layer_ent_means = [np.stack(ent_input_seeds).mean(axis=0)]
layer_labels    = ["Input→H1"]
for k in range(n_internal):
    layer_ent_means.append(np.stack(ent_internal_seeds[k]).mean(axis=0))
    layer_labels.append(f"H{k+1}→H{k+2}")

heatmap = np.stack(layer_ent_means)   # (n_layers, H)
order   = np.argsort(heatmap[0])      # sort by input-layer entropy
heatmap = heatmap[:, order]

fig, ax = plt.subplots(figsize=(10, 2.5))
im = ax.imshow(heatmap, aspect="auto", cmap="Blues_r", origin="upper")
ax.set_yticks(range(len(layer_labels)))
ax.set_yticklabels(layer_labels, fontsize=12)
ax.set_xlabel("Node index (sorted by input-layer φ entropy)", fontsize=12)
ax.set_xticks([])
ax.set_title(f"DHS L3 — posterior φ entropy per node per layer  (N={HEAT_N})", fontsize=13)
cbar = plt.colorbar(im, ax=ax, label="Mean entropy (lower = more concentrated)")
plt.tight_layout()
plt.show()